# Problem 3: Face Detection and Recognition Pipeline

It covers:
- Detector comparison (Haar, MTCNN, optional RetinaFace)
- Alignment + augmentation visualization
- ArcFace vs Softmax training/evaluation
- Task 4A/4B/4C/4D deliverables
- Export of figures and metrics to `outputs/`

In [ ]:
# Section 1: Project setup
%load_ext autoreload
%autoreload 2

import json
import random
from pathlib import Path

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Output dir: {OUT_DIR.resolve()}")

In [ ]:
# Imports from modular files
import cv2
import matplotlib.pyplot as plt

from data_loader import (
    FaceDataset,
    filter_identities,
    load_lfw_arrays,
    set_seed,
    stratified_split,
)
from detectors import (
    HaarDetector,
    MTCNNDetector,
    RetinaFaceDetector,
    build_pseudo_gt,
    evaluate_detector,
)
from alignment import LandmarkAligner, build_aligned_dataset, build_transforms
from losses import ArcFaceLoss, SoftmaxHead
from models import EmbeddingNet
from train import extract_embeddings, train_model
from evaluation import (
    build_identification_protocol,
    evaluate_challenge_drop,
    evaluate_identification,
    evaluate_verification,
    classify_challenge_conditions,
    topk_gallery_matches,
)
from visualization import (
    plot_alignment_examples,
    plot_augmentations,
    plot_challenge_drop,
    plot_cmc_curves,
    plot_detector_comparison,
    plot_roc_curves,
    plot_top5_matches,
    plot_tsne_embeddings,
)

set_seed(SEED)

### Dataset Loading and Preprocessing

We load the LFW dataset, filter identities, and create train/validation/test splits.

**File used:** data_loader.py

In [ ]:
# Load and preprocess
all_images, all_labels, ds = load_lfw_arrays()
print(f"Loaded {len(all_images)} images")

# Keep identities with enough samples for train/val/test + gallery/probe usage
f_images, f_labels = filter_identities(all_images, all_labels, min_images_per_id=7)
splits = stratified_split(f_images, f_labels, seed=SEED)

aligner = LandmarkAligner(device=DEVICE)
aligned_imgs, unaligned_imgs, aligned_labels, failed = build_aligned_dataset(
    splits.train_images + splits.val_images + splits.test_images,
    [str(x) for x in splits.label_encoder.inverse_transform(np.concatenate([splits.train_labels, splits.val_labels, splits.test_labels]))],
    aligner,
)
print(f"Aligned samples: {len(aligned_imgs)} | failed: {failed}")

train_tf, eval_tf = build_transforms()

# Re-encode aligned labels with a shared encoder
from sklearn.preprocessing import LabelEncoder
le2 = LabelEncoder().fit(aligned_labels)
encoded = le2.transform(aligned_labels)

from sklearn.model_selection import train_test_split
idx = np.arange(len(aligned_imgs))
tr_idx, temp_idx = train_test_split(idx, test_size=0.2, stratify=encoded, random_state=SEED)
va_idx, te_idx = train_test_split(temp_idx, test_size=0.5, stratify=encoded[temp_idx], random_state=SEED)

aligned_train_ds = FaceDataset([aligned_imgs[i] for i in tr_idx], encoded[tr_idx], transform=train_tf)
aligned_val_ds = FaceDataset([aligned_imgs[i] for i in va_idx], encoded[va_idx], transform=eval_tf)
aligned_test_ds = FaceDataset([aligned_imgs[i] for i in te_idx], encoded[te_idx], transform=eval_tf)

unaligned_train_ds = FaceDataset([unaligned_imgs[i] for i in tr_idx], encoded[tr_idx], transform=train_tf)

n_classes = len(le2.classes_)
print(f"Classes: {n_classes} | train={len(aligned_train_ds)} val={len(aligned_val_ds)} test={len(aligned_test_ds)}")

### Task 1: Face Detection Pipeline

We compare classical and deep learning-based face detectors:
- Haar Cascade
- MTCNN / RetinaFace

Metrics evaluated:
- Precision, Recall, F1-score
- IoU
- Inference time

**File used:** detectors.py

In [ ]:
# Detector comparison preview (Haar / MTCNN / RetinaFace optional)
haar = HaarDetector()
mtcnn = MTCNNDetector(device=DEVICE)

retina = None
try:
    retina = RetinaFaceDetector()
except Exception as exc:
    print(f"RetinaFace unavailable: {exc}")

sample_img = all_images[0]
det_map = {
    "Haar": haar.detect(sample_img).boxes,
    "MTCNN": mtcnn.detect(sample_img).boxes,
}
if retina is not None:
    det_map["RetinaFace"] = retina.detect(sample_img).boxes

plot_detector_comparison(sample_img, det_map, output_path=str(OUT_DIR / "detector_comparison.png"))

# Detector benchmark on pseudo-GT
eval_subset = all_images[:500]
gt_list = build_pseudo_gt(eval_subset, detector=mtcnn, conf_threshold=0.99)
haar_metrics = evaluate_detector(haar.detect, eval_subset, gt_list, label="Haar")
mtcnn_metrics = evaluate_detector(mtcnn.detect, eval_subset, gt_list, label="MTCNN")
print(haar_metrics)
print(mtcnn_metrics)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for m, ax, color in zip([haar_metrics, mtcnn_metrics],
                         [axes[0], axes[1]], ["steelblue", "coral"]):
    confs = np.array(m["all_confs"])
    ious  = np.array(m["all_ious"])
    if len(confs) == 0:
        continue
    sort_idx = np.argsort(confs)
    ax.scatter(confs[sort_idx], ious[sort_idx],
               alpha=0.3, s=10, color=color)
    # rolling mean
    bins = np.linspace(0, 1, 20)
    bin_means = [ious[(confs >= bins[i]) & (confs < bins[i+1])].mean()
                 if np.any((confs >= bins[i]) & (confs < bins[i+1])) else np.nan
                 for i in range(len(bins) - 1)]
    ax.plot(bins[:-1] + 0.025, bin_means, "k-", lw=2, label="Bin mean IoU")
    ax.set_title(f"{m['label']}: Confidence vs IoU"); ax.set_xlabel("Confidence")
    ax.set_ylabel("IoU"); ax.set_ylim(0, 1); ax.legend()

# Bar chart comparison
metrics_names = ["Precision", "Recall", "F1", "Mean IoU"]
haar_vals  = [haar_metrics[k] for k in ["precision","recall","f1","mean_iou"]]
mtcnn_vals = [mtcnn_metrics[k] for k in ["precision","recall","f1","mean_iou"]]
x = np.arange(len(metrics_names)); width = 0.35
axes[2].bar(x - width/2, haar_vals,  width, label="Haar",  color="steelblue")
axes[2].bar(x + width/2, mtcnn_vals, width, label="MTCNN", color="coral")
axes[2].set_xticks(x); axes[2].set_xticklabels(metrics_names)
axes[2].set_title("Detector Comparison"); axes[2].legend()
axes[2].set_ylim(0, 1.1)

plt.tight_layout(); plt.savefig("task1_detection_metrics.png", dpi=120)
plt.show(); print("Saved: task1_detection_metrics.png")

### Task 2: Face Alignment and Preprocessing

### Alignment Visualization

We compare unaligned and aligned faces to observe the effect of landmark-based alignment.

In [ ]:
# Alignment + augmentation visual outputs
plot_alignment_examples(
    unaligned_imgs=unaligned_imgs,
    aligned_imgs=aligned_imgs,
    n_examples=5,
    output_path=str(OUT_DIR / "alignment_examples.png"),
)

rgb0 = cv2.cvtColor(aligned_imgs[0], cv2.COLOR_BGR2RGB)
plot_augmentations(
    image_rgb=rgb0,
    train_transform=train_tf,
    n_examples=3,
    output_path=str(OUT_DIR / "augmentation_examples.png"),
)

### Task 3: Face Recognition with Metric Learning

We train a deep face recognition model using:
- Backbone: MobileNetV2
- Loss: ArcFace (Angular Margin Loss)

We also compare with a softmax baseline.

**Files used:** models.py, losses.py, train.py

In [ ]:
# Train ArcFace and Softmax baseline
embedding_dim = 512
epochs = 30

arc_model = EmbeddingNet(embedding_dim=embedding_dim)
arc_loss = ArcFaceLoss(embedding_dim=embedding_dim, n_classes=n_classes)
arc_model, arc_hist = train_model(
    model=arc_model,
    loss_module=arc_loss,
    train_ds=aligned_train_ds,
    val_ds=aligned_val_ds,
    device=DEVICE,
    epochs=epochs,
    label="ArcFace",
)

soft_model = EmbeddingNet(embedding_dim=embedding_dim)
soft_loss = SoftmaxHead(embedding_dim=embedding_dim, n_classes=n_classes)
soft_model, soft_hist = train_model(
    model=soft_model,
    loss_module=soft_loss,
    train_ds=aligned_train_ds,
    val_ds=aligned_val_ds,
    device=DEVICE,
    epochs=epochs,
    label="Softmax",
)

arc_unaligned_model = EmbeddingNet(embedding_dim=embedding_dim)
arc_unaligned_loss = ArcFaceLoss(embedding_dim=embedding_dim, n_classes=n_classes)
arc_unaligned_model, _ = train_model(
    model=arc_unaligned_model,
    loss_module=arc_unaligned_loss,
    train_ds=unaligned_train_ds,
    val_ds=aligned_val_ds,
    device=DEVICE,
    epochs=epochs,
    label="ArcFace-Unaligned",
)

# Save checkpoints
torch.save(arc_model.state_dict(), OUT_DIR / "arcface_model.pt")
torch.save(soft_model.state_dict(), OUT_DIR / "softmax_model.pt")

In [ ]:
# Extract test embeddings for evaluation
arc_emb, arc_lbl = extract_embeddings(arc_model, aligned_test_ds, device=DEVICE)
soft_emb, soft_lbl = extract_embeddings(soft_model, aligned_test_ds, device=DEVICE)

assert np.array_equal(arc_lbl, soft_lbl), "Label mismatch between models"
test_labels = arc_lbl
print(arc_emb.shape, test_labels.shape)

### Task 4A: Verification (6,000 pairs)

In [ ]:
arc_ver = evaluate_verification(arc_emb, test_labels, n_same=3000, n_diff=3000, seed=SEED)
soft_ver = evaluate_verification(soft_emb, test_labels, n_same=3000, n_diff=3000, seed=SEED)

print("ArcFace:", arc_ver)
print("Softmax:", soft_ver)

plot_roc_curves(
    [
        {"label": "ArcFace", "fpr": arc_ver.fpr, "tpr": arc_ver.tpr, "auc": arc_ver.auc, "eer": arc_ver.eer},
        {"label": "Softmax", "fpr": soft_ver.fpr, "tpr": soft_ver.tpr, "auc": soft_ver.auc, "eer": soft_ver.eer},
    ],
    output_path=str(OUT_DIR / "task4a_roc.png"),
)

task4a_table = {
    "ArcFace": {
        "AUC": arc_ver.auc,
        "EER": arc_ver.eer,
        "TAR@FAR=0.01": arc_ver.tar_far_001,
        "TAR@FAR=0.001": arc_ver.tar_far_0001,
    },
    "Softmax": {
        "AUC": soft_ver.auc,
        "EER": soft_ver.eer,
        "TAR@FAR=0.01": soft_ver.tar_far_001,
        "TAR@FAR=0.001": soft_ver.tar_far_0001,
    },
}
print(json.dumps(task4a_table, indent=2))

### Task 4B: Identification (100-ID gallery, 50-ID probe)

In [ ]:
gallery_idx, probe_idx = build_identification_protocol(
    test_labels,
    gallery_identities=100,
    gallery_per_id=5,
    probe_identities=50,
    probe_per_id=2,
    seed=SEED,
)

arc_ident = evaluate_identification(arc_emb, test_labels, gallery_idx, probe_idx, max_rank=10)
soft_ident = evaluate_identification(soft_emb, test_labels, gallery_idx, probe_idx, max_rank=10)

print("ArcFace Rank-1/Rank-5:", arc_ident.rank1, arc_ident.rank5)
print("Softmax Rank-1/Rank-5:", soft_ident.rank1, soft_ident.rank5)

plot_cmc_curves(
    [
        {"label": "ArcFace", "cmc": arc_ident.cmc},
        {"label": "Softmax", "cmc": soft_ident.cmc},
    ],
    max_rank=10,
    output_path=str(OUT_DIR / "task4b_cmc.png"),
)

### Task 4C: Natural LFW challenge subsets (no synthetic corruption)

In [ ]:
# Build landmarks/boxes for test images to classify natural conditions
inv_test_images = [aligned_imgs[i] for i in te_idx]
landmarks = []
boxes = []
for img in inv_test_images:
    lm = aligner.get_landmarks(img)
    landmarks.append(lm)
    if lm is not None:
        x, y, w, h = cv2.boundingRect(lm.astype(np.int32))
        boxes.append((x, y, w, h))
    else:
        boxes.append(None)

challenge_split = classify_challenge_conditions(inv_test_images, landmarks, boxes)
challenge_results = evaluate_challenge_drop(
    embeddings=arc_emb,
    labels=test_labels,
    gallery_indices=gallery_idx,
    probe_indices=probe_idx,
    challenge_split=challenge_split,
)

print(json.dumps(challenge_results, indent=2))
plot_challenge_drop(challenge_results, output_path=str(OUT_DIR / "task4c_challenges.png"))

### Task 4D: Embedding space (t-SNE, 20+ identities)

In [ ]:
plot_tsne_embeddings(
    embeddings=arc_emb,
    labels=test_labels,
    n_identities=20,
    output_path=str(OUT_DIR / "task4d_tsne.png"),
)

# Compactness/separation diagnostics
from collections import defaultdict

by_id = defaultdict(list)
for emb, lbl in zip(arc_emb, test_labels):
    by_id[int(lbl)].append(emb)

intra = []
centroids = {}
for k, vals in by_id.items():
    arr = np.vstack(vals)
    c = arr.mean(axis=0)
    c = c / (np.linalg.norm(c) + 1e-9)
    centroids[k] = c
    if len(arr) > 1:
        sims = arr @ c
        intra.append(float(1.0 - sims.mean()))

keys = list(centroids.keys())
inter = []
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        inter.append(float(1.0 - np.dot(centroids[keys[i]], centroids[keys[j]])))

print({
    "mean_intra_distance": float(np.mean(intra)) if intra else None,
    "mean_inter_centroid_distance": float(np.mean(inter)) if inter else None,
})

### Export deliverables (figures, metrics, top-5 matches)

In [ ]:
# Top-5 match visualization for 5 probes
examples = topk_gallery_matches(
    embeddings=arc_emb,
    labels=test_labels,
    gallery_indices=gallery_idx,
    probe_indices=probe_idx,
    k=5,
    n_examples=5,
    seed=SEED,
)
plot_top5_matches(examples, inv_test_images, output_path=str(OUT_DIR / "task4_top5_matches.png"))

metrics = {
    "task4a": task4a_table,
    "task4b": {
        "ArcFace": {"rank1": arc_ident.rank1, "rank5": arc_ident.rank5},
        "Softmax": {"rank1": soft_ident.rank1, "rank5": soft_ident.rank5},
    },
    "task4c": challenge_results,
    "protocol": {
        "gallery_size": int(arc_ident.gallery_size),
        "probe_size": int(arc_ident.probe_size),
    },
}

with open(OUT_DIR / "metrics_summary.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("Saved metrics and figures to outputs/")